# Tool Calls Tombstones

This notebook demonstrates the concept of **tombstones** in tool calls to reduce token consumption in LLM applications.

**Concept**: When tool calls return very long outputs, we can replace them with summarized versions after their first use, reducing token consumption by > 90%

**Source**: Mentioned in a recent [Anthropic podcast on YouTube.](https://www.youtube.com/watch?v=XuvKFsktX0Q)

**Approach Demonstrated**:
**LLM-Derived**: Using a samller LLM to summarize tool output

## Setup


install dependencies

In [1]:
%pip install openai tiktoken



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Initial Tool Call Example


In [2]:
from openai import OpenAI
import tiktoken
import json
import os

## Helper Function
Used to count tokens

In [3]:
encoding = tiktoken.encoding_for_model("gpt-5")

def count_tokens(text):
    return len(encoding.encode(text))

In [4]:
# Prompt for API key securely
api_key = os.getenv("OPENAI_API_KEY")

# Initialize the OpenAI client
client = OpenAI(api_key=api_key)

In [5]:
# Define a tool that returns a very long response (simulating verbose tool outputs)
tools = [
    {
        "type": "function",
        "name": "get_customer_profile",
        "description": "Retrieve detailed customer profile and interaction history from the CRM system.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {
                    "type": "string",
                    "description": "The unique identifier for the customer in the CRM system",
                },
            },
            "required": ["customer_id"],
        },
    },
]

def get_customer_profile(customer_id):
    return f"""CUSTOMER PROFILE - ID: {customer_id}

    BASIC INFORMATION:
    Name: Sarah Chen
    Email: sarah.chen@techstartup.io
    Phone: +1 (555) 234-9876
    Company: TechStartup Inc.
    Position: VP of Engineering
    Industry: Software Development
    Account Value: $485,000 ARR
    Customer Since: March 15, 2023
    Account Status: Active - Premium Tier

    ENGAGEMENT HISTORY:
    Sarah has been an exceptionally engaged customer with our platform, demonstrating consistent usage patterns and a deep understanding of our product capabilities. Over the past 18 months, she has participated in 12 webinars, attended 3 in-person conferences, and completed our advanced certification program. Her team of 45 engineers has achieved a 94% platform adoption rate, which is significantly above our customer average of 67%.

    RECENT INTERACTIONS:
    - Nov 8, 2024: Submitted feature request for enhanced API rate limiting controls
    - Oct 22, 2024: Participated in Beta program for new analytics dashboard
    - Oct 15, 2024: Attended Q3 Business Review meeting with our Customer Success team
    - Sep 30, 2024: Raised support ticket #45892 regarding integration issues (resolved within 4 hours)
    - Sep 12, 2024: Provided testimonial for case study on DevOps transformation

    PURCHASE HISTORY:
    The account has shown steady growth with strategic upsells aligned to their business needs. Initial purchase was our Professional plan at $15,000/month. Six months in, they upgraded to Enterprise at $28,000/month. Most recently, they added our Advanced Security module ($12,500/month) and AI-powered monitoring suite ($8,500/month). Contract renewal is scheduled for March 2025 with strong indicators for expansion into our new Data Governance offering.

    SUPPORT METRICS:
    Total Tickets: 23 (18 resolved, 5 ongoing)
    Average Resolution Time: 6.2 hours
    Customer Satisfaction Score: 9.2/10
    Net Promoter Score: 9/10
    Last Support Contact: 6 days ago

    KEY OPPORTUNITIES:
    Sarah mentioned in our last QBR that they're planning a major infrastructure modernization project in Q1 2025, which could be an excellent opportunity to introduce our Infrastructure Automation suite. Additionally, they're expanding their team by 30% next quarter, which aligns perfectly with our volume licensing incentives. She's also expressed interest in our upcoming Machine Learning Operations module during the beta program feedback session.

    RISK FACTORS:
    Minimal churn risk identified. The only concern noted was the recent integration issue, but it was resolved quickly and Sarah expressed satisfaction with our response time. Competitor analysis shows that two of their portfolio companies use alternative solutions, but Sarah has been vocal about the superior ROI they've experienced with our platform.

    RELATIONSHIP STRENGTH:
    Executive sponsorship is strong with quarterly touchpoints at the C-level. Sarah has introduced us to three other companies in her network, resulting in two new customers. She's scheduled to speak at our annual user conference next month, showcasing their success story on reducing deployment times by 75% using our platform."""


In [6]:
# Initialize conversation with a user query
messages = [
    {"role": "user", "content": "Can you pull up the profile for customer CUST-2847?"}
]

In [7]:
# Make API call with tool definitions
response = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=messages
)

messages += response

# Execute tool calls and append results to messages
for item in response.output:
    if item.type == "function_call":
        if item.name == "get_customer_profile":
            args = json.loads(item.arguments)
            customer_data = get_customer_profile(args["customer_id"])
            
            messages.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                  "customer_profile": customer_data
                })
            })

## Analyze Original Tool Call

Save the original tool call and measure its token count to establish a baseline.


In [8]:

original_tool_call = messages[-1]

In [9]:
original_tool_call

{'type': 'function_call_output',
 'call_id': 'call_IB3PUGpJZJe6Yqof2kuqD6dg',
 'output': '{"customer_profile": "CUSTOMER PROFILE - ID: CUST-2847\\n\\n    BASIC INFORMATION:\\n    Name: Sarah Chen\\n    Email: sarah.chen@techstartup.io\\n    Phone: +1 (555) 234-9876\\n    Company: TechStartup Inc.\\n    Position: VP of Engineering\\n    Industry: Software Development\\n    Account Value: $485,000 ARR\\n    Customer Since: March 15, 2023\\n    Account Status: Active - Premium Tier\\n\\n    ENGAGEMENT HISTORY:\\n    Sarah has been an exceptionally engaged customer with our platform, demonstrating consistent usage patterns and a deep understanding of our product capabilities. Over the past 18 months, she has participated in 12 webinars, attended 3 in-person conferences, and completed our advanced certification program. Her team of 45 engineers has achieved a 94% platform adoption rate, which is significantly above our customer average of 67%.\\n\\n    RECENT INTERACTIONS:\\n    - Nov 8, 20

In [10]:
original_tool_call_tokens = count_tokens(original_tool_call["output"])

In [11]:
original_tool_call_tokens

708

## Tombstone creation

Use a lightweight LLM to automatically summarize the tool output into 5 words or less.

In [12]:
from pydantic import BaseModel, Field

class ToolCallSummary(BaseModel):
    tool_call_name: str = Field(description="The name of the tool call")
    tool_call_summary: str = Field(description="A detailed summary of the tool call output, 2 sentences or less. Keep all unique detail to retain the original meaning.")

In [13]:

# Use a lightweight model to summarize the output
new_output = client.responses.parse(
    model="gpt-5-nano",
    input=original_tool_call["output"],
    instructions="Summarize the tool call output.",
    reasoning={"effort": "minimal"},
    text_format=ToolCallSummary
)

In [14]:
llm_summary_tool_call = new_output.output_parsed

In [15]:
llm_summary_tool_call

ToolCallSummary(tool_call_name='CustomerProfileSummary', tool_call_summary='The profile highlights Sarah Chen, VP of Engineering at TechStartup Inc., a Premium Active customer ($485k ARR) since March 2023. Engagement is high with strong adoption (45 engineers, 94% adoption), multiple events and certifications. Recent interactions include feature request, beta participation, a QBR, a resolved support ticket, and a testimonial. Purchase history shows gradual upsell path: Professional ($15k/mo) to Enterprise ($28k/mo), plus Advanced Security ($12.5k/mo) and AI monitoring ($8.5k/mo); renewal due March 2025 with expansion potential into Data Governance. Support metrics are solid (23 tickets, 9.2/10 CSAT, NPS 9, avg. 6.2h resolution). Key opportunities: Q1 2025 infrastructure modernization with Infrastructure Automation, team expansion (30%), interest in ML Ops module. Low churn risk but past integration issue noted; competitor use exists among portfolio companies. Relationship strength is h

In [16]:
new_llm_summary_tool_call = {
    "type": "function_call_output",
    "call_id": item.call_id,
    "output": json.dumps({
      "customer_profile": f"[TOOL CALL SUMMARY] {llm_summary_tool_call.tool_call_name}: {llm_summary_tool_call.tool_call_summary}"
    })
}

In [17]:
new_llm_summary_tool_call

{'type': 'function_call_output',
 'call_id': 'call_IB3PUGpJZJe6Yqof2kuqD6dg',
 'output': '{"customer_profile": "[TOOL CALL SUMMARY] CustomerProfileSummary: The profile highlights Sarah Chen, VP of Engineering at TechStartup Inc., a Premium Active customer ($485k ARR) since March 2023. Engagement is high with strong adoption (45 engineers, 94% adoption), multiple events and certifications. Recent interactions include feature request, beta participation, a QBR, a resolved support ticket, and a testimonial. Purchase history shows gradual upsell path: Professional ($15k/mo) to Enterprise ($28k/mo), plus Advanced Security ($12.5k/mo) and AI monitoring ($8.5k/mo); renewal due March 2025 with expansion potential into Data Governance. Support metrics are solid (23 tickets, 9.2/10 CSAT, NPS 9, avg. 6.2h resolution). Key opportunities: Q1 2025 infrastructure modernization with Infrastructure Automation, team expansion (30%), interest in ML Ops module. Low churn risk but past integration issue no

In [18]:
new_llm_summary_tool_call_tokens = count_tokens(new_llm_summary_tool_call["output"])

In [19]:
new_llm_summary_tool_call_tokens

234

## Replace in the messages

In [20]:
# Add the summarized tool call back to the messages
messages[-1] = new_llm_summary_tool_call


In [21]:
messages

[{'role': 'user',
  'content': 'Can you pull up the profile for customer CUST-2847?'},
 ('id', 'resp_0952d6a6e07988a2006903ba2380d4819789d0bfb66f80bf98'),
 ('created_at', 1761851939.0),
 ('error', None),
 ('incomplete_details', None),
 ('instructions', None),
 ('metadata', {}),
 ('model', 'gpt-5-2025-08-07'),
 ('object', 'response'),
 ('output',
  [ResponseReasoningItem(id='rs_0952d6a6e07988a2006903ba23fe248197814e617f23a93c1f', summary=[], type='reasoning', content=None, encrypted_content=None, status=None),
   ResponseFunctionToolCall(arguments='{"customer_id":"CUST-2847"}', call_id='call_IB3PUGpJZJe6Yqof2kuqD6dg', name='get_customer_profile', type='function_call', id='fc_0952d6a6e07988a2006903ba25dad481979d05f3394bf16cc7', status='completed')]),
 ('parallel_tool_calls', True),
 ('temperature', 1.0),
 ('tool_choice', 'auto'),
 ('tools',
  [FunctionTool(name='get_customer_profile', parameters={'type': 'object', 'properties': {'customer_id': {'type': 'string', 'description': 'The uniqu

## Results Comparison

Compare token counts between the original and tombstoned tool calls.


In [22]:
# Display original vs tombstoned token counts
(original_tool_call_tokens, new_llm_summary_tool_call_tokens)

(708, 234)

In [23]:
# Calculate percentage reduction
(original_tool_call_tokens - new_llm_summary_tool_call_tokens) / original_tool_call_tokens

0.6694915254237288